# Clase 10: Validación Cruzada y Optimización Bayesiana con Optuna

**Machine Learning (RIA-805) — Bloque 2 — Factibilidad Técnico-Financiera**

**Docente:** Marco Capio  
**Fecha:** Semana 2 del bloque comprimido

**Objetivo:** Aplicar validación cruzada estratificada y optimización bayesiana con Optuna para encontrar la configuración de hiperparámetros que minimiza el costo total del modelo.

**Conexión con la práctica:** En P2 aplicarán esto a su modelo de Wall-Following y exportarán la mejor configuración a `config.json`.

## ¿Dónde estamos?

En la Clase 9 encontramos un umbral de decisión que reduce el costo de los errores. Sin embargo, el modelo base y sus hiperparámetros seguían siendo los mismos. Hoy exploraremos si podemos reducir aún más el costo al optimizar esos hiperparámetros, evaluando cada configuración de forma más estable.

## El problema del split único

Al dividir una sola vez los datos, el resultado puede depender mucho de qué observaciones quedaron en entrenamiento y cuáles en prueba. Una partición favorable puede producir métricas altas; otra partición puede mostrar un rendimiento distinto, aunque el algoritmo no haya cambiado.

> **El modelo no cambió: cambió el split.**

**Solución: validación cruzada.** Repetimos el entrenamiento y la evaluación sobre distintas particiones para obtener una estimación menos dependiente de una sola división.

## Validación cruzada K-Fold

K-Fold divide los datos en `K` partes (*folds*). En cada iteración se reserva una parte para validar y se entrenan los restantes `K-1` folds. Al final se promedian las métricas.

```text
Iteración      Fold 1   Fold 2   Fold 3   Fold 4   Fold 5
     1          VALID   entren.  entren.  entren.  entren.
     2          entren. VALID   entren.  entren.  entren.
     3          entren. entren.  VALID   entren.  entren.
     4          entren. entren.  entren.  VALID   entren.
     5          entren. entren.  entren.  entren.  VALID
```

En **K-Fold estratificado**, cada fold conserva aproximadamente las proporciones de las clases, algo esencial para un problema desbalanceado. Se usan con frecuencia `K=5` o `K=10`.

- **Ventajas:** métricas más estables y todos los datos participan en entrenamiento y validación.
- **Desventajas:** se realizan `K` entrenamientos por configuración, por lo que aumenta el tiempo de cómputo.

## ¿Qué son los hiperparámetros?

Son decisiones de configuración que se establecen antes de entrenar. **No se aprenden de los datos** como los parámetros internos del modelo.

| Modelo | Hiperparámetros comunes | Efecto aproximado |
|---|---|---|
| Random Forest | `n_estimators`, `max_depth`, `min_samples_split` | Número y complejidad de árboles; regularización |
| XGBoost | `n_estimators`, `learning_rate`, `max_depth`, `subsample` | Número de etapas, tasa de aprendizaje y complejidad |
| SVM | `C`, `kernel`, `gamma` | Penalización, forma del límite y alcance del kernel |

Una configuración demasiado simple puede subajustar; una demasiado compleja puede sobreajustar y consumir más recursos.

## Métodos de búsqueda

| Método | Cómo busca | Ventajas | Desventajas |
|---|---|---|---|
| Grid Search | Evalúa todas las combinaciones de una grilla | Exhaustivo dentro de la grilla; sencillo de explicar | El número de combinaciones crece rápidamente |
| Random Search | Prueba combinaciones aleatorias | Aprovecha mejor un presupuesto limitado cuando hay muchos parámetros | Puede repetir pruebas poco útiles y no aprende de resultados anteriores |
| Optimización bayesiana | Usa resultados anteriores para proponer nuevas pruebas | Busca adaptativamente y puede ser eficiente | Requiere definir una función objetivo y controlar el proceso |

## Optuna: optimización bayesiana

Optuna es una biblioteca para buscar hiperparámetros. La **función objetivo** recibe una prueba (*trial*), propone valores y devuelve una métrica que se desea maximizar o minimizar. El muestreador TPE (Tree-structured Parzen Estimator) usa los resultados previos para sugerir combinaciones prometedoras.

El ciclo es: **sugerir parámetros → entrenar/evaluar → registrar resultado → sugerir adaptativamente**. Como referencia didáctica, con 20–50 pruebas Optuna suele encontrar configuraciones competitivas frente a Grid Search con cientos de combinaciones; no es una garantía y depende del problema, presupuesto y espacio de búsqueda.

In [ ]:
# Importamos utilidades para datos, visualización, persistencia y optimización.
import json
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.base import clone

# Configuramos las gráficas, la semilla y el nivel de mensajes de Optuna.
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", module="optuna")
RANDOM_STATE = 42

# Detectamos la raíz tanto si Jupyter se abrió desde la raíz como desde notebooks/.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "processed" / "crop.parquet").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró data/processed/crop.parquet desde la carpeta actual."
    )

# Creamos las carpetas donde se escribirán reportes, figuras y modelos.
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
for output_dir in (REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")

## Cargar datos y preparar el problema binario

Reutilizaremos el problema de la Clase 9: predecir si la observación corresponde a **`rice` (positivo)** o a **cualquier otro cultivo (negativo)**. En el dataset, `rice` es una clase minoritaria frente al conjunto de los demás cultivos, por lo que debemos conservar sus proporciones al dividir los datos.

In [ ]:
# Leemos el dataset procesado y creamos la etiqueta binaria de la Clase 9.
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "crop.parquet")
if "label" not in df.columns:
    raise KeyError("El parquet debe contener la columna objetivo 'label'.")

df = df.copy()
df["target_binario"] = (df["label"] == "rice").astype(int)
X = df.drop(columns=["label", "target_binario"])
y = df["target_binario"]

# Reservamos un test estratificado; el escalador se ajusta solo con train.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Mostramos conteos y proporciones para verificar el desbalance y la estratificación.
distribucion = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).rename(index={0: "no rice (0)", 1: "rice (1)"})
print(f"Observaciones: {len(df)} | Variables predictoras: {X.shape[1]}")
print("Proporciones por clase:")
display(distribucion)

## Validación cruzada del modelo base

Antes de optimizar, veamos cómo rinde el modelo base con validación cruzada. Esto nos da una línea base para comparar. Usaremos F1 porque combina precisión y recall y ayuda a evaluar la detección de la clase minoritaria. El `Pipeline` vuelve a ajustar `RobustScaler` dentro de cada fold, evitando que estadísticas de validación se filtren al entrenamiento.

In [ ]:
# Definimos el modelo base y la validación estratificada reproducible.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
modelo_base = RandomForestClassifier(random_state=RANDOM_STATE)
pipeline_base = Pipeline([
    ("scaler", RobustScaler()),
    ("classifier", clone(modelo_base)),
])

# Medimos F1 en cada fold usando únicamente el conjunto de entrenamiento.
f1_base_folds = cross_val_score(
    pipeline_base, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1
)
f1_base_cv = float(f1_base_folds.mean())
f1_base_std = float(f1_base_folds.std())

# Calculamos también el F1 de una sola partición de prueba como referencia didáctica.
modelo_base_split = clone(modelo_base)
modelo_base_split.fit(X_train_scaled, y_train)
y_pred_base = modelo_base_split.predict(X_test_scaled)
f1_base_split = f1_score(y_test, y_pred_base)
print(f"F1 CV (5 folds): {f1_base_cv:.4f} ± {f1_base_std:.4f}")
print(f"F1 split único de test (referencia de Clase 8): {f1_base_split:.4f}")

## Definir costos

Recordemos los costos de la Clase 9: **C_FP = $50** por un falso positivo y **C_FN = $500** por un falso negativo. Perder un caso positivo (`rice`) cuesta diez veces más que activar una intervención innecesaria. En este ejercicio, el umbral de clasificación se mantiene en el valor predeterminado; optimizamos hiperparámetros, no el umbral.

In [ ]:
def calcular_costo_cv(modelo, X, y, C_FP=50, C_FN=500, cv=5):
    """Calcula el costo medio por fold de un clasificador en validación cruzada.

    El costo de cada fold es FP*C_FP + FN*C_FN, usando el umbral
    predeterminado del clasificador. Si `cv` es un entero, se construye
    StratifiedKFold con cinco folds y semilla fija.
    """
    splitter = (
        StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
        if isinstance(cv, int)
        else cv
    )
    costos = []
    for train_idx, valid_idx in splitter.split(X, y):
        modelo_fold = clone(modelo)
        X_fold_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
        X_fold_valid = X.iloc[valid_idx] if hasattr(X, "iloc") else X[valid_idx]
        y_fold_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
        y_fold_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]
        modelo_fold.fit(X_fold_train, y_fold_train)
        predicciones = modelo_fold.predict(X_fold_valid)
        tn, fp, fn, tp = confusion_matrix(
            y_fold_valid, predicciones, labels=[0, 1]
        ).ravel()
        costos.append(int(fp) * C_FP + int(fn) * C_FN)
    return float(np.mean(costos))

## Costo del modelo base con validación cruzada

Ahora calculamos el costo promedio del modelo base con validación cruzada. Esta será nuestra línea base económica. También compararemos con el costo en el split de test de la Clase 9; los valores no son directamente equivalentes porque uno promedia folds y el otro corresponde a una partición.

In [ ]:
# Evaluamos el pipeline base en folds y el modelo base en el split de test.
C_FP = 50
C_FN = 500
costo_base_cv = calcular_costo_cv(
    pipeline_base, X_train, y_train, C_FP=C_FP, C_FN=C_FN, cv=cv
)
tn_base_test, fp_base_test, fn_base_test, tp_base_test = confusion_matrix(
    y_test, y_pred_base, labels=[0, 1]
).ravel()
costo_base_split = int(fp_base_test) * C_FP + int(fn_base_test) * C_FN
print(f"Costo base promedio CV: ${costo_base_cv:,.2f} por fold")
print(f"Costo base en split único de test: ${costo_base_split:,.2f}")
print(f"Errores en test: FP={fp_base_test}, FN={fn_base_test}")

## Introducción a Optuna

Optuna administra un **estudio** que ejecuta varios *trials*. En cada trial la función objetivo solicita valores al objeto `trial`, construye y evalúa un modelo y devuelve un número. La dirección del estudio (`maximize` o `minimize`) indica qué resultado buscamos. Los resultados se registran para que el muestreador TPE guíe pruebas posteriores.

## Función objetivo: maximizar F1

Primero definiremos una función objetivo que maximiza F1. Esto nos permitirá comparar la búsqueda de una métrica tradicional con la búsqueda basada directamente en costo. Cada configuración se evalúa con los mismos cinco folds estratificados.

In [ ]:
def objective_f1(trial):
    """Devuelve el F1 medio de una configuración en cinco folds estratificados."""
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
    }
    pipeline = Pipeline([
        ("scaler", RobustScaler()),
        ("classifier", RandomForestClassifier(
            **params, random_state=RANDOM_STATE, n_jobs=1
        )),
    ])
    scores = cross_val_score(
        pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=1
    )
    return float(scores.mean())

In [ ]:
# Ejecutamos la búsqueda orientada a F1 con un presupuesto de 20 trials.
study_f1 = optuna.create_study(direction="maximize")
study_f1.optimize(objective_f1, n_trials=20)
print("Mejores hiperparámetros (F1):", study_f1.best_params)
print(f"Mejor F1 medio CV: {study_f1.best_value:.4f}")

## Función objetivo: minimizar costo

Ahora definimos la función objetivo más importante para este caso: minimizar el costo total. La función usa exactamente los mismos rangos de hiperparámetros y los mismos folds que la búsqueda de F1, pero evalúa FP y FN con sus costos. La métrica optimizada debe representar la prioridad real del problema.

In [ ]:
def objective_costo(trial):
    """Devuelve el costo medio por fold para los hiperparámetros propuestos."""
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
    }
    pipeline = Pipeline([
        ("scaler", RobustScaler()),
        ("classifier", RandomForestClassifier(
            **params, random_state=RANDOM_STATE, n_jobs=1
        )),
    ])
    return calcular_costo_cv(
        pipeline, X_train, y_train, C_FP=C_FP, C_FN=C_FN, cv=cv
    )

In [ ]:
# Buscamos la configuración de menor costo en el mismo espacio de búsqueda.
study_costo = optuna.create_study(direction="minimize")
study_costo.optimize(objective_costo, n_trials=20)
print("Mejores hiperparámetros (costo):", study_costo.best_params)
print(f"Mejor costo medio CV: ${study_costo.best_value:,.2f} por fold")

## Comparación de resultados

Comparemos el modelo base, la configuración seleccionada por F1 y la seleccionada por costo. Para que la comparación sea justa, los F1 y costos de las configuraciones optimizadas se vuelven a medir en los mismos folds de entrenamiento. El conjunto de test se mantiene reservado para la evaluación final.

In [ ]:
# Construimos los pipelines de las dos mejores configuraciones encontradas.
def crear_pipeline(params):
    """Crea un pipeline con escalado por fold y un Random Forest reproducible."""
    return Pipeline([
        ("scaler", RobustScaler()),
        ("classifier", RandomForestClassifier(
            **params, random_state=RANDOM_STATE, n_jobs=1
        )),
    ])

pipeline_mejor_f1 = crear_pipeline(study_f1.best_params)
pipeline_mejor_costo = crear_pipeline(study_costo.best_params)
f1_mejor_f1_cv = float(cross_val_score(
    pipeline_mejor_f1, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1
).mean())
costo_mejor_f1_cv = calcular_costo_cv(
    pipeline_mejor_f1, X_train, y_train, C_FP=C_FP, C_FN=C_FN, cv=cv
)
f1_mejor_costo_cv = float(cross_val_score(
    pipeline_mejor_costo, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1
).mean())
costo_mejor_costo_cv = calcular_costo_cv(
    pipeline_mejor_costo, X_train, y_train, C_FP=C_FP, C_FN=C_FN, cv=cv
)

# Guardamos la tabla de comparación de entrenamiento en la carpeta de reportes.
comparacion = pd.DataFrame([
    {
        "nombre": "modelo base",
        "n_estimators": modelo_base.n_estimators,
        "max_depth": modelo_base.max_depth,
        "F1 (CV)": f1_base_cv,
        "costo (CV)": costo_base_cv,
    },
    {
        "nombre": "mejor F1",
        "n_estimators": study_f1.best_params["n_estimators"],
        "max_depth": study_f1.best_params["max_depth"],
        "F1 (CV)": f1_mejor_f1_cv,
        "costo (CV)": costo_mejor_f1_cv,
    },
    {
        "nombre": "mejor costo",
        "n_estimators": study_costo.best_params["n_estimators"],
        "max_depth": study_costo.best_params["max_depth"],
        "F1 (CV)": f1_mejor_costo_cv,
        "costo (CV)": costo_mejor_costo_cv,
    },
])
comparacion.to_csv(REPORTS_DIR / "comparacion_optuna.csv", index=False)
print(comparacion.to_string(index=False, float_format=lambda valor: f"{valor:.4f}"))
print(f"Reporte guardado en: {REPORTS_DIR / 'comparacion_optuna.csv'}")

## Análisis de resultados

Observa qué configuración presenta el menor costo y si coincide con la que obtuvo mayor F1. No siempre coinciden: F1 pondera precisión y recall de forma armónica, mientras que la función de costo penaliza explícitamente los falsos negativos diez veces más que los falsos positivos. La elección final depende del impacto del error en el contexto de uso.

## Visualizar la historia de optimización

Optuna permite inspeccionar cómo evoluciona la métrica trial a trial. En esta gráfica mostramos el costo de cada prueba y el mejor costo alcanzado hasta ese punto. Un costo menor es mejor; los primeros trials pueden ser exploratorios antes de que TPE concentre la búsqueda.

In [ ]:
# Graficamos los costos registrados y la mejor solución acumulada por trial.
trials_completados = [
    trial for trial in study_costo.trials
    if trial.value is not None
]
numeros_trial = [trial.number for trial in trials_completados]
valores_costo = [trial.value for trial in trials_completados]
mejor_hasta_trial = np.minimum.accumulate(valores_costo)

fig, ax = plt.subplots()
ax.scatter(numeros_trial, valores_costo, alpha=0.65, label="Costo por trial")
ax.plot(numeros_trial, mejor_hasta_trial, color="darkorange",
        linewidth=2, label="Mejor costo acumulado")
ax.set(title="Historia de optimización del costo",
       xlabel="Número de trial", ylabel="Costo medio por fold ($)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "optuna_history.png", dpi=150, bbox_inches="tight")
plt.show()

## Importancia de hiperparámetros

La importancia estima qué parámetros explican más variación en los resultados observados del estudio. Es una ayuda para interpretar esta búsqueda, no una relación causal ni una garantía de que el mismo orden se mantendrá con otros datos o rangos.

In [ ]:
# Calculamos y graficamos la importancia de los parámetros del estudio de costo.
importancias = optuna.importance.get_param_importances(study_costo)
importancias_df = pd.DataFrame({
    "hiperparámetro": list(importancias.keys()),
    "importancia": list(importancias.values()),
}).sort_values("importancia", ascending=False)

fig, ax = plt.subplots()
sns.barplot(
    data=importancias_df,
    x="importancia",
    y="hiperparámetro",
    color="steelblue",
    ax=ax,
)
ax.set(title="Importancia de hiperparámetros (estudio de costo)",
       xlabel="Importancia relativa", ylabel="")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "optuna_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## Entrenar el modelo final optimizado

Entrenamos el modelo final con los mejores hiperparámetros encontrados al minimizar costo y lo evaluamos en el conjunto de test, que no participó en la búsqueda. Reportaremos F1, falsos positivos, falsos negativos y costo. La evaluación en test es una estimación final; no debe usarse para seguir ajustando repetidamente la búsqueda.

In [ ]:
# Ajustamos el escalador ya definido solo sobre train y entrenamos el modelo final.
modelo_final = RandomForestClassifier(
    **study_costo.best_params, random_state=RANDOM_STATE, n_jobs=1
)
modelo_final.fit(X_train_scaled, y_train)
y_pred_final = modelo_final.predict(X_test_scaled)
f1_final_test = f1_score(y_test, y_pred_final)
tn_final, fp_final, fn_final, tp_final = confusion_matrix(
    y_test, y_pred_final, labels=[0, 1]
).ravel()
costo_final_test = int(fp_final) * C_FP + int(fn_final) * C_FN

# Comparamos en el mismo test contra el modelo base entrenado en el split.
print(f"F1 modelo optimizado en test: {f1_final_test:.4f}")
print(f"FP={fp_final} | FN={fn_final}")
print(f"Costo optimizado en test: ${costo_final_test:,.2f}")
print(f"Costo modelo base en el mismo test: ${costo_base_split:,.2f}")

## Guardar configuración

Guardamos los hiperparámetros, las métricas de validación cruzada y las métricas de test en `config.json` para que la decisión sea reproducible. También serializamos el modelo entrenado. El escalador usado para el modelo final se guarda junto al clasificador para aplicar la misma transformación en inferencia.

In [ ]:
# Guardamos parámetros, costos, métricas y el estado reproducible del experimento.
config = {
    "modelo": "RandomForestClassifier",
    "objetivo_optimizacion": "minimizar costo",
    "mejores_hiperparametros": study_costo.best_params,
    "random_state": RANDOM_STATE,
    "costos": {"C_FP": C_FP, "C_FN": C_FN},
    "validacion_cruzada": {
        "n_splits": 5,
        "costo_medio_por_fold": costo_mejor_costo_cv,
        "f1_medio": f1_mejor_costo_cv,
    },
    "test": {
        "f1": float(f1_final_test),
        "costo": int(costo_final_test),
        "fp": int(fp_final),
        "fn": int(fn_final),
    },
}
config_path = PROJECT_ROOT / "config.json"
with config_path.open("w", encoding="utf-8") as config_file:
    json.dump(config, config_file, indent=2, ensure_ascii=False)

# Guardamos juntos el clasificador y el escalador ajustado con train.
modelo_path = MODELS_DIR / "random_forest_optimizado.pkl"
joblib.dump({"scaler": scaler, "model": modelo_final}, modelo_path)
print(json.dumps(config, indent=2, ensure_ascii=False))
print(f"Configuración guardada en: {config_path}")
print(f"Modelo y escalador guardados en: {modelo_path}")

## Cuándo usar Optuna y cuándo no

| Situación | Enfoque recomendado | Motivo |
|---|---|---|
| Pocos hiperparámetros y rangos pequeños | Grid Search | La búsqueda exhaustiva sigue siendo manejable |
| Muchos hiperparámetros o evaluaciones costosas | Optuna | La búsqueda adaptativa aprovecha resultados previos |
| Modelo destinado a producción | Optuna + CV + función de costo | Alinea la selección con la estabilidad y el impacto operativo |

**Regla práctica:** si vas a pasar más de 30 minutos ajustando hiperparámetros a mano, usa Optuna. Ajusta el número de trials al tiempo y recursos disponibles; más trials no siempre significan una mejora relevante.

## Conclusiones

1. La validación cruzada estratificada reduce la dependencia de un split único y conserva la proporción de las clases.
2. Optuna usa los resultados de trials anteriores para proponer configuraciones de manera adaptativa.
3. La función objetivo define qué significa “mejor” para el problema.
4. Maximizar F1 no es necesariamente lo mismo que minimizar el costo económico.
5. Guardar hiperparámetros, métricas, escalador y modelo facilita la reproducibilidad y el despliegue.

**Próxima clase:** fundamentos de aprendizaje por refuerzo (RL) y Gymnasium.

In [ ]:
# Cerramos la actividad con la ubicación de los artefactos generados.
print("Clase 10 completada. Revisa reports/figures/ y config.json")